# 0. ENVIRONMENT SETUP & OPTIMIZATIONS

In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

# GPU Check
gpus = tf.config.list_physical_devices('GPU')
print("GPUs Available:", gpus)
if gpus:
    try:
        # Aktifkan memory growth agar tidak langsung mengambil semua memori
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Aktifkan XLA (JIT compilation) untuk percepat operasi
        tf.config.optimizer.set_jit(True)
        print(" XLA JIT enabled.")
    except RuntimeError as e:
        print(e)

# Mengaktifkan Mixed Precision (FP16)
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("Compute dtype:", policy.compute_dtype)
print("Variable dtype:", policy.variable_dtype)

import numpy as np
import pandas as pd
import json
import os
import zipfile
import datetime
import shutil

# Seed
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# TensorBoard
import tensorboard
print("TensorBoard version:", tensorboard.__version__)

2026-05-25 08:55:08.474934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779699308.729455      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779699308.800443      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779699309.342915      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779699309.342953      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779699309.342956      24 computation_placer.cc:177] computation placer alr

TensorFlow version: 2.19.0
GPUs Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
 XLA JIT enabled.
Compute dtype: float16
Variable dtype: float32
TensorBoard version: 2.19.0


# 1. Extract Processed Dataset

In [2]:
PROCESSED_INPUT_DIR = "/kaggle/input/datasets/gilangagung/qlop-intelligent-learning-recommendation-dataset/QLOP Recomendation System Dataset"
PROCESSED_DIR = "/kaggle/working/processed"

if os.path.exists(PROCESSED_INPUT_DIR):
    if os.path.exists(PROCESSED_DIR):
        shutil.rmtree(PROCESSED_DIR)
        
    shutil.copytree(PROCESSED_INPUT_DIR, PROCESSED_DIR)
    print("Processed data copied to working directory.")
    print("Files:", os.listdir(PROCESSED_DIR))
else:
    print(f"Error: Folder {PROCESSED_INPUT_DIR} not found.")
    
print("Processed data Copied.")
print("Files:", os.listdir(PROCESSED_DIR))


Processed data copied to working directory.
Files: ['train_test_split_info.json', 'skill_vocab.json', 'synthetic_demand_course_model4.npz', 'synthetic_demand_course_model4.csv', 'synthetic_users_model3.npz', 'role_freq.json', 'unique_skills_raw.txt']
Processed data Copied.
Files: ['train_test_split_info.json', 'skill_vocab.json', 'synthetic_demand_course_model4.npz', 'synthetic_demand_course_model4.csv', 'synthetic_users_model3.npz', 'role_freq.json', 'unique_skills_raw.txt']


# 2. Load Vocabulary And Data Synthetis

In [3]:
# --- Vocabulary ---
vocab_path = os.path.join(PROCESSED_DIR, "skill_vocab.json")
with open(vocab_path, 'r') as f:
    vocab_data = json.load(f)
skill_to_idx = vocab_data["skill_to_idx"]
idx_to_skill = vocab_data["idx_to_skill"]
N_SKILLS = vocab_data["vocab_size"]
print(f"Vocabulary size: {N_SKILLS}")

# --- Synthetic users (Model 3) ---
users_npz = np.load(os.path.join(PROCESSED_DIR, "synthetic_users_model3.npz"))
X_users = users_npz["X_users"]          # (20000, N_SKILLS)
X_roles_idx = users_npz["X_roles"]      # (20000,)
Y_targets = users_npz["Y_targets"]      # (20000, N_SKILLS)
role_list = users_npz["role_list"]      # list
N_ROLES = len(role_list)
role_to_idx = {role: i for i, role in enumerate(role_list)}
idx_to_role = {i: role for role, i in role_to_idx.items()}
print(f"Users: {X_users.shape}, Roles: {N_ROLES}")

# --- Demand-Course data (Model 4) ---
demand_npz = np.load(os.path.join(PROCESSED_DIR, "synthetic_demand_course_model4.npz"))
demand_vectors = demand_npz["demand_vectors"]   # (15000, N_SKILLS)
course_vectors_all = demand_npz["course_vectors"] # semua course
course_indices = demand_npz["course_indices"]   # (15000,)
targets_demand = demand_npz["targets"]          # (15000,)
course_names = demand_npz["course_names"]
course_urls = demand_npz["course_urls"]
num_courses = len(course_names)
print(f"Demand: {demand_vectors.shape}, Courses: {num_courses}")

# --- Split info ---
split_path = os.path.join(PROCESSED_DIR, "train_test_split_info.json")
with open(split_path, 'r') as f:
    split_info = json.load(f)
train_idx_u = split_info["model3"]["train_indices"]
val_idx_u = split_info["model3"]["val_indices"]
train_idx_d = split_info["model4"]["train_indices"]
val_idx_d = split_info["model4"]["val_indices"]

Vocabulary size: 582
Users: (20000, 582), Roles: 50
Demand: (15000, 582), Courses: 1980


# 3. Custom Loss, Callback, & Layers

In [4]:

# --- Weighted MAE ---
class WeightedMAE(tf.keras.losses.Loss):
    def __init__(self, alpha=2.0, name="weighted_mae"):
        super().__init__(name=name)
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        weights = 1.0 + self.alpha * y_true
        absolute_error = tf.abs(y_true - y_pred)
        weighted_error = weights * absolute_error
        return tf.reduce_mean(weighted_error)

# --- EarlyStoppingOnTargetMAE ---
class EarlyStoppingOnTargetMAE(tf.keras.callbacks.Callback):
    def __init__(self, target_mae=0.02, save_path="./best_model"):
        super().__init__()
        self.target_mae = target_mae
        self.save_path = save_path
        self.best_mae = np.inf
        self.stopped = False    

    def on_epoch_end(self, epoch, logs=None):
        current_mae = logs.get("val_mae")
        if current_mae is not None:
            if current_mae < self.best_mae:
                self.best_mae = current_mae
                self.model.export(self.save_path)
                print(f"\n Model improved at epoch {epoch+1} (MAE: {current_mae:.4f}). Saved.")
            if current_mae <= self.target_mae:
                print(f"\n Target MAE {self.target_mae} reached at epoch {epoch+1}. Stopping training.")
                self.model.stop_training = True
                
# --- Interaction Layer for Two-Tower ---
class InteractionLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim=32):
        super(InteractionLayer, self).__init__()
        self.embed_dim = embed_dim
        self.w = self.add_weight(
            shape=(embed_dim, embed_dim),
            initializer='random_normal',
            trainable=True,
            name='interaction_weight'
        )

    def call(self, demand_emb, course_emb):
        # Pastikan float32 untuk operasi (mixed policy bisa float16, kita cast)
        demand_emb = tf.cast(demand_emb, tf.float32)
        course_emb = tf.cast(course_emb, tf.float32)
        demand_transformed = tf.matmul(demand_emb, self.w)
        interaction = tf.reduce_sum(demand_transformed * course_emb, axis=1, keepdims=True)
        return interaction

print("Custom Loss, Callback, and Layer Done")

Custom Loss, Callback, and Layer Done


# 4. SKILL PRIORITY SCORER (Model 3) Build & Train dengan Optimasi Pipeline

In [5]:

# --- Arsitektur (Functional API) ---
def build_model3(N_SKILLS, N_ROLES, embedding_dim=8):
    input_skills = tf.keras.layers.Input(shape=(N_SKILLS,), dtype=tf.float32, name="user_skills")
    input_role = tf.keras.layers.Input(shape=(1,), dtype=tf.int32, name="role_index")
    
    role_emb = tf.keras.layers.Embedding(input_dim=N_ROLES, output_dim=embedding_dim, name="role_embedding")(input_role)
    role_emb = tf.keras.layers.Flatten()(role_emb)
    
    concat = tf.keras.layers.Concatenate()([input_skills, role_emb])
    x = tf.keras.layers.Dense(128, activation='relu', name="dense1")(concat)
    x = tf.keras.layers.Dropout(0.3, name="dropout1")(x)
    x = tf.keras.layers.Dense(64, activation='relu', name="dense2")(x)
    # Output: float32 untuk stabilitas, aktivasi sigmoid
    output = tf.keras.layers.Dense(N_SKILLS, activation='sigmoid', dtype=tf.float32, name="output")(x)
    
    model = tf.keras.Model(inputs=[input_skills, input_role], outputs=output, name="Model3_PriorityScorer")
    return model

model3 = build_model3(N_SKILLS, N_ROLES)
model3.summary()

# --- Optimized Data Pipeline ---
BATCH_SIZE = 64
X_train_skills = X_users[train_idx_u]
X_train_roles = X_roles_idx[train_idx_u].reshape(-1, 1)
y_train = Y_targets[train_idx_u]

X_val_skills = X_users[val_idx_u]
X_val_roles = X_roles_idx[val_idx_u].reshape(-1, 1)
y_val = Y_targets[val_idx_u]

train_ds = tf.data.Dataset.from_tensor_slices(((X_train_skills, X_train_roles), y_train))
train_ds = train_ds.shuffle(10000).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(((X_val_skills, X_val_roles), y_val))
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

# --- Mixed Precision Loss Optimizer ---
loss_fn = WeightedMAE(alpha=2.0)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# Bungkus dengan LossScaleOptimizer untuk mixed precision
optimizer = tf.keras.mixed_precision.LossScaleOptimizer(optimizer)

# Metrics
train_loss = tf.keras.metrics.Mean(name='train_loss')
train_mae = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
train_acc = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='train_acc')
val_loss = tf.keras.metrics.Mean(name='val_loss')
val_mae = tf.keras.metrics.MeanAbsoluteError(name='val_mae')
val_acc = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='val_acc')

# TensorBoard
log_dir = "/kaggle/working/logs/model3/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
summary_writer = tf.summary.create_file_writer(log_dir)

# Callback
os.makedirs("/kaggle/working/models/model3_savedmodel", exist_ok=True)
early_stop3 = EarlyStoppingOnTargetMAE(target_mae=0.02, save_path="/kaggle/working/models/model3_savedmodel")

# --- Training Functions (dengan @tf.function) ---
@tf.function
def train_step3(x, y):
    with tf.GradientTape() as tape:
        predictions = model3(x, training=True)
        loss = loss_fn(y, predictions)
        # Jika mixed precision, loss perlu di-scale
        scaled_loss = optimizer.get_scaled_loss(loss) if hasattr(optimizer, 'get_scaled_loss') else loss
    gradients = tape.gradient(scaled_loss, model3.trainable_variables)
    if hasattr(optimizer, 'get_scaled_loss'):
        gradients = optimizer.get_unscaled_gradients(gradients)
    optimizer.apply_gradients(zip(gradients, model3.trainable_variables))
    # Update metrics
    train_loss.update_state(loss)
    train_mae.update_state(y, predictions)
    y_bin = tf.cast(y > 0.5, tf.float32)
    pred_bin = tf.cast(predictions > 0.5, tf.float32)
    train_acc.update_state(y_bin, pred_bin)

def val_step3(x, y):
    predictions = model3(x, training=False)
    loss = loss_fn(y, predictions)
    val_loss.update_state(loss)
    val_mae.update_state(y, predictions)
    y_bin = tf.cast(y > 0.5, tf.float32)
    pred_bin = tf.cast(predictions > 0.5, tf.float32)
    val_acc.update_state(y_bin, pred_bin)

# --- Training Loop ---
EPOCHS = 30
print("Start training Skill Priority Scorer (Model 3)...")
for epoch in range(EPOCHS):
    train_loss.reset_state()
    train_mae.reset_state()
    train_acc.reset_state()
    val_loss.reset_state()
    val_mae.reset_state()
    val_acc.reset_state()
    
    for step, (x_batch, y_batch) in enumerate(train_ds):
        train_step3(x_batch, y_batch)
    
    for x_val_batch, y_val_batch in val_ds:
        val_step3(x_val_batch, y_val_batch)
    
    # Logging
    t_l = train_loss.result().numpy()
    t_m = train_mae.result().numpy()
    t_a = train_acc.result().numpy()
    v_l = val_loss.result().numpy()
    v_m = val_mae.result().numpy()
    v_a = val_acc.result().numpy()
    
    with summary_writer.as_default():
        tf.summary.scalar('loss/train', t_l, step=epoch)
        tf.summary.scalar('mae/train', t_m, step=epoch)
        tf.summary.scalar('accuracy/train', t_a, step=epoch)
        tf.summary.scalar('loss/val', v_l, step=epoch)
        tf.summary.scalar('mae/val', v_m, step=epoch)
        tf.summary.scalar('accuracy/val', v_a, step=epoch)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {t_l:.4f} MAE: {t_m:.4f} Acc: {t_a:.4f} | Val Loss: {v_l:.4f} MAE: {v_m:.4f} Acc: {v_a:.4f}")
    
    early_stop3.set_model(model3)
    early_stop3.on_epoch_end(epoch, {"val_mae": v_m, "val_loss": v_l})
    if early_stop3.stopped:
        print(" Training stopped by callback.")
        break

if not os.path.exists("/kaggle/working/models/model3_savedmodel"):
    model3.export("/kaggle/working/models/model3_savedmodel")
    print(" Model 3 saved (final).")

I0000 00:00:1779699340.548654      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779699340.554608      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "Model3_PriorityScorer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ role_index          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ role_embedding      │ (None, 1, 8)      │        400 │ role_index[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_skills         │ (None, 582)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ role_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 590)       │          0 │ user_skills[0][0… │
│ (Concatenate)       │                   │            │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 128)       │     75,648 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout1 (Dropout)  │ (None, 128)       │          0 │ dense1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense2 (Dense)      │ (None, 64)        │      8,256 │ dropout1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 582)       │     37,830 │ dense2[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 122,134 (477.09 KB)

 Trainable params: 122,134 (477.09 KB)

...

Saved artifact at '/kaggle/working/models/model3_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 582), dtype=tf.float32, name='user_skills'), TensorSpec(shape=(None, 1), dtype=tf.int32, name='role_index')]
Output Type:
  TensorSpec(shape=(None, 582), dtype=tf.float32, name=None)
Captures:
  137977606463632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988622608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988625104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629712: TensorSpec(shape=(), dtype=tf.resource, name=None)

 Model improved at epoch 28 (MAE: 0.0092). Saved.

 Target MAE 0.02 reached at epoch 28. Stopping training.
Epoch 29/30 | Train Loss: 0.01

INFO:tensorflow:Assets written to: /kaggle/working/models/model3_savedmodel/assets


Saved artifact at '/kaggle/working/models/model3_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 582), dtype=tf.float32, name='user_skills'), TensorSpec(shape=(None, 1), dtype=tf.int32, name='role_index')]
Output Type:
  TensorSpec(shape=(None, 582), dtype=tf.float32, name=None)
Captures:
  137977606463632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988622608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988625104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629712: TensorSpec(shape=(), dtype=tf.resource, name=None)

 Model improved at epoch 29 (MAE: 0.0092). Saved.

 Target MAE 0.02 reached at epoch 29. Stopping training.
Epoch 30/30 | Train Loss: 0.01

INFO:tensorflow:Assets written to: /kaggle/working/models/model3_savedmodel/assets


Saved artifact at '/kaggle/working/models/model3_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 582), dtype=tf.float32, name='user_skills'), TensorSpec(shape=(None, 1), dtype=tf.int32, name='role_index')]
Output Type:
  TensorSpec(shape=(None, 582), dtype=tf.float32, name=None)
Captures:
  137977606463632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988622608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988627792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988625104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137973988629712: TensorSpec(shape=(), dtype=tf.resource, name=None)

 Model improved at epoch 30 (MAE: 0.0092). Saved.

 Target MAE 0.02 reached at epoch 30. Stopping training.


# 5. Two Tower Course Recommendation Model - Training

In [6]:
class InteractionLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim=32):
        super(InteractionLayer, self).__init__()
        self.embed_dim = embed_dim
        self.w = self.add_weight(
            shape=(embed_dim, embed_dim),
            initializer='random_normal',
            trainable=True,
            name='interaction_weight',
            dtype=tf.float32         
        )

    def call(self, demand_emb, course_emb):
        demand_emb = tf.cast(demand_emb, tf.float32)
        course_emb = tf.cast(course_emb, tf.float32)
        w_float32 = tf.cast(self.w, tf.float32)
        demand_transformed = tf.matmul(demand_emb, w_float32)
        interaction = tf.reduce_sum(demand_transformed * course_emb, axis=1, keepdims=True)
        return interaction

print("Done")

Done


In [7]:

# --- Model Definition ---
class TwoTowerModel(tf.keras.Model):
    def __init__(self, N_SKILLS, embed_dim=32):
        super(TwoTowerModel, self).__init__()
        self.demand_tower = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(N_SKILLS,)),
            tf.keras.layers.Dense(embed_dim)
        ], name="DemandTower")
        self.course_tower = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(N_SKILLS,)),
            tf.keras.layers.Dense(embed_dim)
        ], name="CourseTower")
        self.interaction = InteractionLayer(embed_dim)
        self.output_act = tf.keras.layers.Activation('sigmoid', dtype=tf.float32)
    
    def call(self, inputs):
        demand_vec, course_vec = inputs
        demand_emb = self.demand_tower(demand_vec)
        course_emb = self.course_tower(course_vec)
        score = self.interaction(demand_emb, course_emb)
        return self.output_act(score)

model4 = TwoTowerModel(N_SKILLS, embed_dim=32)
# Tes
sample_d = tf.random.normal((2, N_SKILLS))
sample_c = tf.random.normal((2, N_SKILLS))
print("Output shape:", model4([sample_d, sample_c]).shape)

# --- Optimized Data Pipeline ---
course_vectors_selected = course_vectors_all[course_indices]
targets = targets_demand.reshape(-1, 1)

train_demand = demand_vectors[train_idx_d]
train_course = course_vectors_selected[train_idx_d]
train_targets = targets[train_idx_d]

val_demand = demand_vectors[val_idx_d]
val_course = course_vectors_selected[val_idx_d]
val_targets = targets[val_idx_d]

train_ds4 = tf.data.Dataset.from_tensor_slices(((train_demand, train_course), train_targets))
train_ds4 = train_ds4.shuffle(10000).batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

val_ds4 = tf.data.Dataset.from_tensor_slices(((val_demand, val_course), val_targets))
val_ds4 = val_ds4.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

# --- Loss & Optimizer ---
loss_fn4 = tf.keras.losses.MeanSquaredError()
optimizer4 = tf.keras.optimizers.Adam(learning_rate=0.001)

# Metrics
t_loss4 = tf.keras.metrics.Mean(name='train_loss')
t_mae4 = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
t_acc4 = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='train_acc')
v_loss4 = tf.keras.metrics.Mean(name='val_loss')
v_mae4 = tf.keras.metrics.MeanAbsoluteError(name='val_mae')
v_acc4 = tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='val_acc')

# TensorBoard
log_dir4 = "/kaggle/working/logs/model4/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
writer4 = tf.summary.create_file_writer(log_dir4)

# Callback
os.makedirs("/kaggle/working/models/model4_savedmodel", exist_ok=True)
early_stop4 = EarlyStoppingOnTargetMAE(target_mae=0.02, save_path="/kaggle/working/models/model4_savedmodel")

# --- Training Functions ---
@tf.function
def train_step4(demand, course, y):
    with tf.GradientTape() as tape:
        predictions = model4([demand, course], training=True)
        loss = loss_fn4(y, predictions)
    gradients = tape.gradient(loss, model4.trainable_variables)
    optimizer4.apply_gradients(zip(gradients, model4.trainable_variables))
    t_loss4.update_state(loss)
    t_mae4.update_state(y, predictions)
    y_bin = tf.cast(y > 0.5, tf.float32)
    pred_bin = tf.cast(predictions > 0.5, tf.float32)
    t_acc4.update_state(y_bin, pred_bin)

def val_step4(demand, course, y):
    predictions = model4([demand, course], training=False)
    loss = loss_fn4(y, predictions)
    v_loss4.update_state(loss)
    v_mae4.update_state(y, predictions)
    y_bin = tf.cast(y > 0.5, tf.float32)
    pred_bin = tf.cast(predictions > 0.5, tf.float32)
    v_acc4.update_state(y_bin, pred_bin)

# --- Training Loop ---
print("Start training Model 4...")
for epoch in range(EPOCHS):
    t_loss4.reset_state()
    t_mae4.reset_state()
    t_acc4.reset_state()
    v_loss4.reset_state()
    v_mae4.reset_state()
    v_acc4.reset_state()
    
    for (x_d, x_c), y_batch in train_ds4:
        train_step4(x_d, x_c, y_batch)
    for (x_d, x_c), y_batch in val_ds4:
        val_step4(x_d, x_c, y_batch)
    
    tl4 = t_loss4.result().numpy()
    tm4 = t_mae4.result().numpy()
    ta4 = t_acc4.result().numpy()
    vl4 = v_loss4.result().numpy()
    vm4 = v_mae4.result().numpy()
    va4 = v_acc4.result().numpy()
    
    with writer4.as_default():
        tf.summary.scalar('loss/train', tl4, step=epoch)
        tf.summary.scalar('mae/train', tm4, step=epoch)
        tf.summary.scalar('accuracy/train', ta4, step=epoch)
        tf.summary.scalar('loss/val', vl4, step=epoch)
        tf.summary.scalar('mae/val', vm4, step=epoch)
        tf.summary.scalar('accuracy/val', va4, step=epoch)
    
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {tl4:.4f} MAE: {tm4:.4f} Acc: {ta4:.4f} | Val Loss: {vl4:.4f} MAE: {vm4:.4f} Acc: {va4:.4f}")
    
    early_stop4.set_model(model4)
    early_stop4.on_epoch_end(epoch, {"val_mae": vm4, "val_loss": vl4})
    if early_stop4.stopped:
        print(" Training stopped by callback.")
        break

if not os.path.exists("/kaggle/working/models/model4_savedmodel"):
    model4.export("/kaggle/working/models/model4_savedmodel")
    print(" Model 4 saved.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Output shape: (2, 1)
Start training Model 4...
Epoch  1/30 | Train Loss: 0.0275 MAE: 0.0675 Acc: 0.9975 | Val Loss: 0.0005 MAE: 0.0049 Acc: 1.0000
INFO:tensorflow:Assets written to: /kaggle/working/models/model4_savedmodel/assets


INFO:tensorflow:Assets written to: /kaggle/working/models/model4_savedmodel/assets


Saved artifact at '/kaggle/working/models/model4_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 582), dtype=tf.float16, name=None), TensorSpec(shape=(None, 582), dtype=tf.float16, name=None)]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137973839949520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863634384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863629008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863630736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863633040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863630352: TensorSpec(shape=(), dtype=tf.resource, name=None)

 Model improv

INFO:tensorflow:Assets written to: /kaggle/working/models/model4_savedmodel/assets


Saved artifact at '/kaggle/working/models/model4_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 582), dtype=tf.float16, name=None), TensorSpec(shape=(None, 582), dtype=tf.float16, name=None)]
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  137973839949520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863634384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863629008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863630736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863631312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863633040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137970863630352: TensorSpec(shape=(), dtype=tf.resource, name=None)

 Model improv

# 6. Zip Model and Inference Test

In [8]:
zip_model_path = "/kaggle/working/saved_models.zip"
with zipfile.ZipFile(zip_model_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk("/kaggle/working/models"):
        for file in files:
            full = os.path.join(root, file)
            arcname = os.path.relpath(full, "/kaggle/working/models")
            zf.write(full, arcname)
print(f" Models zipped to {zip_model_path}")

 Models zipped to /kaggle/working/saved_models.zip


In [9]:
# ============================================
# INFERENCE TEST (Fixed dtypes & signature names)
# ============================================
import tensorflow as tf

# --- Load Model 3 ---
loaded3 = tf.saved_model.load("/kaggle/working/models/model3_savedmodel")
infer3 = loaded3.signatures['serving_default']
print("✅ Model 3 loaded.")

# --- Load Model 4 ---
loaded4 = tf.saved_model.load("/kaggle/working/models/model4_savedmodel")
infer4 = loaded4.signatures['serving_default']
print("✅ Model 4 loaded.")

# --- Simulasi User Input ---
cv_skills = ["python", "sql"]
role_input = "Backend Developer"
cv_skills = [s.lower().strip() for s in cv_skills]
print(f"User skills: {cv_skills}, Role: {role_input}")

# ============================================
# Step 1: Prediksi Skill Prioritas (Model 3)
# ============================================
user_vec = np.zeros((1, N_SKILLS), dtype=np.float32)
for s in cv_skills:
    if s in skill_to_idx:
        user_vec[0, skill_to_idx[s]] = 1.0
    else:
        print(f"⚠️ Skill '{s}' not in vocabulary, ignoring.")

role_idx = np.array([[role_to_idx[role_input]]], dtype=np.int32)

out3 = infer3(
    user_skills=tf.constant(user_vec),
    role_index=tf.constant(role_idx)
)
pred_scores = out3['output_0'].numpy()[0]

# Masking
mask = user_vec[0] == 0
pred_scores_masked = np.where(mask, pred_scores, -1.0)

top_indices = np.argsort(pred_scores_masked)[::-1][:10]
top_skills = []
for idx in top_indices:
    if pred_scores_masked[idx] > 0:
        top_skills.append({
            "skill": idx_to_skill[str(idx)],
            "priority_score": float(pred_scores_masked[idx])
        })

print("\n📊 Top-10 Priority Skills to Learn:")
for i, item in enumerate(top_skills, 1):
    print(f"  {i}. {item['skill']:<30s} score: {item['priority_score']:.4f}")

# ============================================
# Step 2: Rekomendasi Kursus (Model 4)
# ============================================
demand_vec = np.zeros((1, N_SKILLS), dtype=np.float32)
for item in top_skills:
    sk = item['skill']
    if sk in skill_to_idx:
        demand_vec[0, skill_to_idx[sk]] = 1.0

demand_batch = np.tile(demand_vec, (num_courses, 1)).astype(np.float32)
course_batch = course_vectors_all.astype(np.float32)

# Ambil nama input dari signature (kita sudah lihat: 'args_0_1' dan 'args_0')
sig4_keys = list(infer4.structured_input_signature[1].keys())
print("🔍 Model 4 input keys:", sig4_keys)

# Cast input ke float16 karena model 4 dilatih dengan mixed precision
demand_batch_f16 = tf.cast(tf.constant(demand_batch), tf.float16)
course_batch_f16 = tf.cast(tf.constant(course_batch), tf.float16)

# Panggil dengan keyword arguments sesuai nama yang muncul
kwargs4 = {
    sig4_keys[0]: demand_batch_f16,
    sig4_keys[1]: course_batch_f16
}
out4 = infer4(**kwargs4)
match_scores = list(out4.values())[0].numpy().flatten()
top_course_idx = np.argsort(match_scores)[::-1][:20]

print("\n🎓 Top-20 Recommended Courses:")
for i, cid in enumerate(top_course_idx, 1):
    score = match_scores[cid]
    name = course_names[cid]
    url = course_urls[cid]
    covered = [idx_to_skill[str(k)] for k in range(N_SKILLS)
               if course_vectors_all[cid][k] > 0 and demand_vec[0][k] > 0]
    print(f"  {i:2d}. {name:<60s} score: {score:.4f} | covered: {covered}")

print("\n✅ Inference test completed.")

✅ Model 3 loaded.
✅ Model 4 loaded.
User skills: ['python', 'sql'], Role: Backend Developer
⚠️ Skill 'sql' not in vocabulary, ignoring.

📊 Top-10 Priority Skills to Learn:
  1. post                           score: 0.3390
  2. c                              score: 0.2279
  3. google                         score: 0.2248
  4. reduce                         score: 0.2091
  5. programming language           score: 0.1489
  6. git                            score: 0.1385
  7. linkedin                       score: 0.1347
  8. linux                          score: 0.1244
  9. docker                         score: 0.1238
  10. javascript                     score: 0.1177
🔍 Model 4 input keys: ['args_0_1', 'args_0']

🎓 Top-20 Recommended Courses:
   1. Machine Learning and Data Analytics Part 1                   score: 0.0218 | covered: []
   2. Microsoft Back-End Developer                                 score: 0.0218 | covered: []
   3. Data Analysis and Visualization with Power BI          

In [10]:
print("TensorBoard logs:")
print(f"  Model 3: {log_dir}")
print(f"  Model 4: {log_dir4}")
print("Launch in notebook with: %tensorboard --logdir /kaggle/working/logs")

TensorBoard logs:
  Model 3: /kaggle/working/logs/model3/20260525-085543
  Model 4: /kaggle/working/logs/model4/20260525-085645
Launch in notebook with: %tensorboard --logdir /kaggle/working/logs
